# CS781 Phase 3 — Experiment 4
## Phase-1 Model 3 (Encoder–Decoder) + Prefix Tuning — Rhetorical Role Prediction

**Experiment identity:** the exact Phase-1 **Model 3 encoder–decoder checkpoint** is loaded strictly and frozen.  
For Phase 3, the trainable PEFT mechanism is **true attention Prefix Tuning**: each of the 9 encoder self-attention layers receives learned **prefix key/value vectors**. No ordinary input token is replaced and no pretrained weight is updated.

### Why encoder-side prefix tuning here?
Rhetorical-role prediction is sentence classification. Model 3's Phase-1 encoder is already bidirectional and is the natural classification representation. The complete encoder–decoder checkpoint is still instantiated and strictly loaded; the pretrained decoder remains frozen and unchanged, but is not needed for the classification forward path.

This is **not Prompt Tuning**:
- Prompt Tuning would prepend trainable *input embeddings*.
- This notebook prepends trainable **K/V states inside every encoder attention layer**, which is Prefix Tuning.

### Core Phase-3 contract
- Training data: `CL_train + IT_train` only.
- Dev selection: combined `CL_dev + IT_dev` 13-label Macro-F1.
- Exact supplied Phase-3 evaluator is run before final refit.
- Optional transition decoding is tuned on dev only.
- Fresh train+dev refit replays the exact selected training position.
- Private test is inference-only.
- Heldout, refit, and heldout+refit ensemble submissions are created.
- Every test variant is structurally validated.
- `prediction.json` is at the ZIP root.

### Efficiency / stability
- mid-epoch dev evaluation catches early peaks;
- deterministic sortish length grouping reduces padding;
- cached tokenization;
- fused AdamW where available;
- AMP, live elapsed time and ETA;
- strict guards ensure only prefix parameters + RR task head are trainable.

> Attach `encdec_full_best*.pt` and the exact Phase-1 `tokenizer.json`, keep `HF_TOKEN` in Kaggle Secrets, and run top-to-bottom.

---
### What changed in v2 (core method untouched: frozen Model 3, encoder K/V Prefix Tuning, same head/data/eval contract)
| Area | Change | Why |
|---|---|---|
| **Bug fix** | Cell 14 now defines `records_to_prediction`, writes `cl_dev_truth.json`, `it_dev_truth.json`, `labels.json` and `prediction_dev.json`, and re-checks the reloaded checkpoint's dev F1 | v1 never created these, so Cells 16, 17 and 21 would crash with `NameError` / missing files after training |
| **Bug fix** | Manifest uses `TOKENIZER_FILE` instead of the undefined `TOKENIZER_PATH_RESOLVED` | crash in the last cell |
| Quality | Prefix K/V are **initialised from real K/V activations** of the frozen encoder (sampled token positions of real training inputs) instead of N(0, 0.02) | this scale-matched initialization heuristic starts the learned prefix from frozen encoder K/V states rather than arbitrary tiny random vectors. It remains pure Prefix Tuning because only the learned prefix K/V parameters are updated. Toggle: `PREFIX_INIT_FROM_TOKENS` |
| Reproducibility | Prefix and head are built under fixed seeds in both the dev run and the refit | the refit now starts from exactly the same initialisation the validated run started from |
| Hygiene | Per-label report is printed only when a new best is found (full report every eval was 12 long printouts) | log readability |

---
### v3 review
I kept the useful v2 changes and added only robustness/reproducibility checks.  
The **core experiment is unchanged**: frozen Phase-1 Model 3 + encoder self-attention K/V Prefix Tuning + the same Phase-3 data/evaluator/submission contract.

Additional v3 safeguards:
- activation-based prefix init is verified to use **train records only**;
- prefix tensors are checked for finite values and expected shape after initialization;
- the exact evaluator files created in Cell 14 are asserted to exist before scoring;
- initialization mode/statistics are written to the experiment manifest.


In [1]:
# ============================================================
# CELL 1 — Imports, seed, paths, and Experiment-4 config
# ============================================================
import os, re, gc, json, math, time, random, zipfile, subprocess, sys, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from huggingface_hub import HfApi, hf_hub_download, login

SEED = 1234
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Could not read Kaggle HF_TOKEN secret:", repr(e))
assert HF_TOKEN, "HF_TOKEN not found in Kaggle Secrets."
login(token=HF_TOKEN)
HF_REPO = "Exploration-Lab/CS-781-Capstone"

ROOT = Path("/kaggle/working/phase3_exp4_model3_prefix")
ROOT.mkdir(parents=True, exist_ok=True)
BEST_CKPT = ROOT / "best_prefix_rr.pt"
FINAL_CKPT = ROOT / "final_refit_prefix_rr.pt"
DEV_PRED = ROOT / "prediction_dev.json"
CL_DEV_TRUTH = ROOT / "cl_dev_truth.json"
IT_DEV_TRUTH = ROOT / "it_dev_truth.json"
LABELS_JSON = ROOT / "labels.json"
EVAL_SCRIPT = ROOT / "evaluate_phase3.py"
OFFICIAL_PRED = ROOT / "prediction.json"
SUBMISSION_ZIP = ROOT / "submission.zip"

# -------- Exact Phase-1 Model 3 architecture --------
VOCAB_SIZE = 24_576
D_MODEL = 512
N_ENCODER_LAYERS = 9
N_DECODER_LAYERS = 9
N_HEADS = 8
HEAD_DIM = D_MODEL // N_HEADS
FFN_DIM = 2048
DROPOUT = 0.0
MAX_ENCODER_POSITIONS = 2048
MAX_DECODER_POSITIONS = 1536
EXPECTED_TOKENIZER_SHA256 = "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

# -------- Phase-3 context --------
MAX_LEN = 384
CONTEXT_SENTS = 1
USE_POS = True
TARGET_BUDGET = 250
PREV_BUDGET = 55
NEXT_BUDGET = 55

# -------- True Prefix Tuning --------
PREFIX_LEN = 20
PREFIX_DROPOUT = 0.10
PREFIX_LR = 3e-4
# Direct learned K/V prefixes per encoder layer:
# 2 * layers * prefix_len * d_model = 184,320 trainable prefix scalars.
PREFIX_INIT_STD = 0.02          # only used when PREFIX_INIT_FROM_TOKENS = False
PREFIX_INIT_FROM_TOKENS = True  # init prefix K/V from real frozen-encoder K/V activations
PREFIX_INIT_RECORDS = 256       # training records sampled for that init
PREFIX_INIT_MODE = "train_activation_kv" if PREFIX_INIT_FROM_TOKENS else "gaussian_random"

# -------- RR head --------
# Encoder is bidirectional already, so a heavy BiGRU is unnecessary here.
# Target/global pooling + a compact MLP keeps this experiment focused on Prefix Tuning.
HEAD_DROPOUT = 0.20
HEAD_HIDDEN = 384
HEAD_LR = 7e-4

# -------- Optimization --------
NUM_EPOCHS = 6
EVAL_PER_EPOCH = 2
EARLY_STOP_PATIENCE_EVALS = 4
MICRO_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 24
GRAD_ACCUM_STEPS = 2
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_RATIO = 0.06
USE_AMP = True
LABEL_SMOOTHING = 0.03
CLASS_WEIGHT_MIN = 0.45
CLASS_WEIGHT_MAX = 2.50

PRINT_EVERY = 250
NUM_WORKERS = 2
LENGTH_GROUPED_TRAINING = True
LENGTH_BUCKET_MULT = 64
NAN_CHECK_EVERY = 50
VERBOSE_EVAL_REPORT = False   # True = full per-label report at every eval (default: only for new bests)

REFIT_MATCH_ORIGINAL_SCHEDULE = True
ENSEMBLE_HELDOUT_AND_REFIT = True

print("Output directory:", ROOT)
print(f"Model3: encoder={N_ENCODER_LAYERS} decoder={N_DECODER_LAYERS} | d={D_MODEL} | heads={N_HEADS} | FFN={FFN_DIM}")
print(f"Prefix Tuning: encoder self-attention K/V | prefix_len={PREFIX_LEN} | dropout={PREFIX_DROPOUT}")
print("Phase-1 decoder: strictly loaded and frozen; classification uses encoder representations")
print("Context sentences each side:", CONTEXT_SENTS, "| MAX_LEN:", MAX_LEN)


Device: cuda
GPU: Tesla T4
Output directory: /kaggle/working/phase3_exp4_model3_prefix
Model3: encoder=9 decoder=9 | d=512 | heads=8 | FFN=2048
Prefix Tuning: encoder self-attention K/V | prefix_len=20 | dropout=0.1
Phase-1 decoder: strictly loaded and frozen; classification uses encoder representations
Context sentences each side: 1 | MAX_LEN: 384


In [2]:
# ============================================================
# CELL 2 — Find Phase-1 Model 3 checkpoint and tokenizer safely
# ============================================================
def find_files(predicate):
    out=[]
    for base in ["/kaggle/input", "/kaggle/working"]:
        if not os.path.isdir(base): continue
        for root, _, files in os.walk(base):
            for fn in files:
                p=os.path.join(root,fn)
                if predicate(fn,p): out.append(p)
    return out

ckpt_matches = find_files(
    lambda fn,p: fn.lower().endswith(".pt") and "encdec_full_best" in fn.lower()
)
if not ckpt_matches:
    ckpt_matches = find_files(
        lambda fn,p: fn.lower().endswith(".pt")
        and "encdec" in p.lower()
        and "best" in fn.lower()
    )
tok_matches = find_files(
    lambda fn,p: fn.lower().endswith(".json") and "tokenizer" in fn.lower()
)

print("Checkpoint candidates:")
for p in ckpt_matches: print(" ", p)
print("Tokenizer candidates:")
for p in tok_matches: print(" ", p)

assert ckpt_matches, "Attach the Phase-1 Model3 best checkpoint (encdec_full_best*.pt)."
assert tok_matches, "Attach the exact Phase-1 tokenizer.json."

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20), b""):
            h.update(b)
    return h.hexdigest()

def score_path(p, words):
    low=p.lower()
    return sum(w in low for w in words)

PHASE1_CKPT = sorted(
    ckpt_matches,
    key=lambda p:(score_path(p,["model3","encdec","phase1","encdec_full_best"]),p),
    reverse=True
)[0]

exact_tok=[p for p in tok_matches if sha256_file(p)==EXPECTED_TOKENIZER_SHA256]
TOKENIZER_FILE = exact_tok[0] if exact_tok else sorted(tok_matches)[0]

print("Using checkpoint:", PHASE1_CKPT)
print("Using tokenizer :", TOKENIZER_FILE)
print("Tokenizer SHA256:", sha256_file(TOKENIZER_FILE))
assert sha256_file(TOKENIZER_FILE)==EXPECTED_TOKENIZER_SHA256, (
    "Tokenizer hash mismatch — attach the exact Phase-1 tokenizer."
)


Checkpoint candidates:
  /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/encdec_full_best (2).pt
Tokenizer candidates:
  /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/tokenizer (1).json
Using checkpoint: /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/encdec_full_best (2).pt
Using tokenizer : /kaggle/input/datasets/aaryant23/phase2-model3-ckpts/tokenizer (1).json
Tokenizer SHA256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [3]:
# ============================================================
# CELL 3 — Original custom byte-level LegalBPE
# ============================================================

BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            tok.symbol_bytes[s] = bytes([b])
        for l, r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
print("Tokenizer vocab:", len(tokenizer))
assert len(tokenizer) == VOCAB_SIZE, (len(tokenizer), VOCAB_SIZE)

probe = "IN THE SUPREME COURT OF INDIA\nSection 14: legal test."
assert tokenizer.decode(tokenizer.encode(probe)) == probe
print("✅ tokenizer round-trip passed")


Tokenizer vocab: 24576
✅ tokenizer round-trip passed


In [4]:
# ============================================================
# CELL 4 — Exact Phase-1 Model 3 architecture
#           + prefix-capable encoder attention path
# ============================================================
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)
    def forward(self, position_ids):
        return self.pe[position_ids]


class EncoderSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, key_valid=None, prefix_kv=None):
        B,T,C=x.shape
        qkv=self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v=qkv.unbind(dim=2)
        q,k,v=q.transpose(1,2),k.transpose(1,2),v.transpose(1,2)

        p_len=0
        if prefix_kv is not None:
            pk,pv=prefix_kv
            # Prefix tensors may be shared across the batch: (1,H,P,D)
            if pk.size(0)==1 and B>1:
                pk=pk.expand(B,-1,-1,-1)
                pv=pv.expand(B,-1,-1,-1)
            assert pk.shape[:2]==(B,self.n_heads)
            assert pv.shape==pk.shape
            p_len=pk.size(2)
            k=torch.cat([pk.to(k.dtype),k],dim=2)
            v=torch.cat([pv.to(v.dtype),v],dim=2)

        attn_mask=None
        if key_valid is not None:
            kv=key_valid.bool()
            if p_len:
                pvalid=torch.ones((B,p_len),dtype=torch.bool,device=x.device)
                kv=torch.cat([pvalid,kv],dim=1)
            attn_mask=kv[:,None,None,:]

        y=F.scaled_dot_product_attention(
            q,k,v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.out_proj(y)


class DecoderSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super().__init__()
        assert d_model%n_heads==0
        self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model)
        self.out_proj=nn.Linear(d_model,d_model)
        self.dropout=dropout
    def forward(self,x,past_kv=None,use_cache=False):
        B,T,C=x.shape
        qkv=self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v=qkv.unbind(dim=2)
        q,k,v=q.transpose(1,2),k.transpose(1,2),v.transpose(1,2)
        mask,is_causal=None,True
        if past_kv is not None:
            pk,pv=past_kv
            k=torch.cat([pk,k],dim=2); v=torch.cat([pv,v],dim=2)
            if T==1:
                is_causal=False
            else:
                S=k.size(2); P=S-T
                mask=torch.ones(T,S,device=x.device).tril(diagonal=P).bool()
                is_causal=False
        y=F.scaled_dot_product_attention(
            q,k,v,attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal
        )
        y=y.transpose(1,2).contiguous().view(B,T,C)
        y=self.out_proj(y)
        return y,((k,v) if use_cache else None)


class CrossAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super().__init__()
        assert d_model%n_heads==0
        self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.q_proj=nn.Linear(d_model,d_model)
        self.kv_proj=nn.Linear(d_model,2*d_model)
        self.out_proj=nn.Linear(d_model,d_model)
        self.dropout=dropout
    def forward(self,x,memory,memory_valid=None,cached_kv=None,use_cache=False):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        if cached_kv is None:
            S=memory.size(1)
            kv=self.kv_proj(memory).view(B,S,2,self.n_heads,self.head_dim)
            k,v=kv.unbind(dim=2)
            k,v=k.transpose(1,2),v.transpose(1,2)
        else:
            k,v=cached_kv
        attn_mask=None if memory_valid is None else memory_valid[:,None,None,:]
        y=F.scaled_dot_product_attention(
            q,k,v,attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False
        )
        y=y.transpose(1,2).contiguous().view(B,T,C)
        y=self.out_proj(y)
        return y,((k,v) if use_cache else None)


class EncoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,ffn_dim,dropout):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model)
        self.attn=EncoderSelfAttention(d_model,n_heads,dropout)
        self.ln2=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(
            nn.Linear(d_model,ffn_dim),nn.GELU(),
            nn.Linear(ffn_dim,d_model),nn.Dropout(dropout)
        )
        self.drop=nn.Dropout(dropout)
    def forward(self,x,key_valid=None,prefix_kv=None):
        x=x+self.drop(self.attn(self.ln1(x),key_valid=key_valid,prefix_kv=prefix_kv))
        x=x+self.ff(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,ffn_dim,dropout):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model)
        self.self_attn=DecoderSelfAttention(d_model,n_heads,dropout)
        self.ln2=nn.LayerNorm(d_model)
        self.cross_attn=CrossAttention(d_model,n_heads,dropout)
        self.ln3=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(
            nn.Linear(d_model,ffn_dim),nn.GELU(),
            nn.Linear(ffn_dim,d_model),nn.Dropout(dropout)
        )
        self.drop=nn.Dropout(dropout)
    def forward(self,x,memory,memory_valid=None,past_self_kv=None,cached_cross_kv=None,use_cache=False):
        a,new_self=self.self_attn(self.ln1(x),past_kv=past_self_kv,use_cache=use_cache)
        x=x+self.drop(a)
        c,new_cross=self.cross_attn(
            self.ln2(x),memory,memory_valid=memory_valid,
            cached_kv=cached_cross_kv,use_cache=use_cache
        )
        x=x+self.drop(c)
        x=x+self.ff(self.ln3(x))
        return x,new_self,new_cross


class EncoderDecoderLM(nn.Module):
    def __init__(self,vocab_size,d_model,n_encoder_layers,n_decoder_layers,
                 n_heads,ffn_dim,dropout,max_encoder_positions,max_decoder_positions):
        super().__init__()
        self.vocab_size=vocab_size
        self.d_model=d_model
        self.emb_scale=math.sqrt(d_model)
        self.max_encoder_positions=max_encoder_positions
        self.max_decoder_positions=max_decoder_positions
        self.token_emb=nn.Embedding(vocab_size,d_model)
        self.enc_pos=SinusoidalPositionalEncoding(d_model,max_encoder_positions)
        self.dec_pos=SinusoidalPositionalEncoding(d_model,max_decoder_positions)
        self.drop=nn.Dropout(dropout)
        self.encoder=nn.ModuleList([
            EncoderBlock(d_model,n_heads,ffn_dim,dropout)
            for _ in range(n_encoder_layers)
        ])
        self.decoder=nn.ModuleList([
            DecoderBlock(d_model,n_heads,ffn_dim,dropout)
            for _ in range(n_decoder_layers)
        ])
        self.enc_ln=nn.LayerNorm(d_model)
        self.dec_ln=nn.LayerNorm(d_model)
        self.apply(self._init_weights)

        enc_scaled=0.02/math.sqrt(2*n_encoder_layers)
        dec_scaled=0.02/math.sqrt(3*n_decoder_layers)
        for block in self.encoder:
            nn.init.normal_(block.attn.out_proj.weight,mean=0.0,std=enc_scaled)
            nn.init.normal_(block.ff[2].weight,mean=0.0,std=enc_scaled)
        for block in self.decoder:
            nn.init.normal_(block.self_attn.out_proj.weight,mean=0.0,std=dec_scaled)
            nn.init.normal_(block.cross_attn.out_proj.weight,mean=0.0,std=dec_scaled)
            nn.init.normal_(block.ff[2].weight,mean=0.0,std=dec_scaled)

    def _init_weights(self,m):
        if isinstance(m,nn.Linear):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m,nn.Embedding):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)

    def encode(self,input_ids,key_valid=None,position_ids=None,prefix_key_values=None):
        B,S=input_ids.shape
        if S>self.max_encoder_positions:
            raise ValueError(f"encoder length {S} > {self.max_encoder_positions}")
        if position_ids is None:
            if key_valid is None:
                position_ids=torch.arange(S,device=input_ids.device)[None,:].expand(B,S)
            else:
                position_ids=(key_valid.long().cumsum(1)-1).clamp(min=0)

        x=self.token_emb(input_ids)*self.emb_scale + self.enc_pos(position_ids).to(self.token_emb.weight.dtype)
        x=self.drop(x)
        for i,block in enumerate(self.encoder):
            pkv=None if prefix_key_values is None else prefix_key_values[i]
            x=block(x,key_valid=key_valid,prefix_kv=pkv)
        return self.enc_ln(x)

    def decode_hidden(self,decoder_ids,memory,memory_valid=None,
                      past_self_key_values=None,cross_key_values=None,
                      use_cache=False,position_ids=None):
        B,T=decoder_ids.shape
        past_len=0 if past_self_key_values is None else past_self_key_values[0][0].size(2)
        if past_len+T>self.max_decoder_positions:
            raise ValueError(f"decoder length {past_len+T} > {self.max_decoder_positions}")
        if position_ids is None:
            position_ids=torch.arange(
                past_len,past_len+T,device=decoder_ids.device
            )[None,:].expand(B,T)
        x=self.token_emb(decoder_ids)*self.emb_scale + self.dec_pos(position_ids).to(self.token_emb.weight.dtype)
        x=self.drop(x)
        new_self,new_cross=([] if use_cache else None),([] if use_cache else None)
        for i,block in enumerate(self.decoder):
            ps=None if past_self_key_values is None else past_self_key_values[i]
            ck=None if cross_key_values is None else cross_key_values[i]
            x,s_kv,c_kv=block(
                x,memory,memory_valid=memory_valid,
                past_self_kv=ps,cached_cross_kv=ck,use_cache=use_cache
            )
            if use_cache:
                new_self.append(s_kv); new_cross.append(c_kv)
        return self.dec_ln(x),new_self,new_cross

    def project(self,hidden):
        return F.linear(hidden,self.token_emb.weight)


def build_backbone():
    return EncoderDecoderLM(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        n_encoder_layers=N_ENCODER_LAYERS,
        n_decoder_layers=N_DECODER_LAYERS,
        n_heads=N_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_encoder_positions=MAX_ENCODER_POSITIONS,
        max_decoder_positions=MAX_DECODER_POSITIONS,
    )

backbone=build_backbone()
n_backbone=sum(p.numel() for p in backbone.parameters())
print("Backbone params:",f"{n_backbone/1e6:.3f}M")
assert 75_000_000 <= n_backbone <= 85_000_000, n_backbone


Backbone params: 78.793M


In [5]:
# ============================================================
# CELL 5 — Strictly load Phase-1 Model 3 checkpoint
# ============================================================
obj=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
print("Checkpoint keys:",list(obj.keys()) if isinstance(obj,dict) else type(obj))
state=obj["model"] if isinstance(obj,dict) and "model" in obj else obj

missing,unexpected=backbone.load_state_dict(state,strict=True)
assert not missing and not unexpected
print("✅ strict=True checkpoint load succeeded")

if isinstance(obj,dict):
    print(
        "step:",obj.get("global_step",obj.get("step")),
        "val_loss:",obj.get("val_loss",obj.get("best_val"))
    )
    cfg=obj.get("config",{})
    if cfg:
        print("Checkpoint architecture:",cfg.get("architecture"))
        aliases = {
            "d_model": D_MODEL,
            "n_encoder_layers": N_ENCODER_LAYERS,
            "n_decoder_layers": N_DECODER_LAYERS,
            "n_heads": N_HEADS,
            "ffn_dim": FFN_DIM,
        }
        for k,expected in aliases.items():
            if k in cfg:
                assert int(cfg[k])==int(expected),(k,cfg[k],expected)

backbone=backbone.to(DEVICE)


Checkpoint keys: ['model', 'global_step', 'val_loss', 'config']
✅ strict=True checkpoint load succeeded
step: 16266 val_loss: 1.885548301174893
Checkpoint architecture: encoder_decoder_transformer


In [6]:
# ============================================================
# CELL 6 — True encoder attention Prefix Tuning
# ============================================================
class EncoderKVPrefix(nn.Module):
    """
    Direct learned K/V prefixes for every encoder layer.

    Parameter shapes:
      prefix_key   : [layers, prefix_len, heads, head_dim]
      prefix_value : [layers, prefix_len, heads, head_dim]

    They are inserted as attention K/V states, not as input embeddings.
    """
    def __init__(self,n_layers,prefix_len,n_heads,head_dim,init_std=PREFIX_INIT_STD,dropout=PREFIX_DROPOUT):
        super().__init__()
        self.n_layers=n_layers
        self.prefix_len=prefix_len
        self.n_heads=n_heads
        self.head_dim=head_dim
        self.prefix_key=nn.Parameter(
            torch.empty(n_layers,prefix_len,n_heads,head_dim)
        )
        self.prefix_value=nn.Parameter(
            torch.empty(n_layers,prefix_len,n_heads,head_dim)
        )
        nn.init.normal_(self.prefix_key,mean=0.0,std=init_std)
        nn.init.normal_(self.prefix_value,mean=0.0,std=init_std)
        self.drop=nn.Dropout(dropout)

    def forward(self):
        out=[]
        for i in range(self.n_layers):
            # [P,H,D] -> [1,H,P,D]
            k=self.drop(self.prefix_key[i]).permute(1,0,2).unsqueeze(0)
            v=self.drop(self.prefix_value[i]).permute(1,0,2).unsqueeze(0)
            out.append((k,v))
        return out


# Freeze every original Phase-1 parameter.
for p in backbone.parameters():
    p.requires_grad=False

seed_everything(SEED+1)   # same prefix init in the dev run and in the refit
prefix_module=EncoderKVPrefix(
    N_ENCODER_LAYERS,PREFIX_LEN,N_HEADS,HEAD_DIM
).to(DEVICE)

prefix_params=sum(p.numel() for p in prefix_module.parameters())
expected_prefix=2*N_ENCODER_LAYERS*PREFIX_LEN*D_MODEL
assert prefix_params==expected_prefix,(prefix_params,expected_prefix)

print("Prefix layers:",N_ENCODER_LAYERS)
print("Prefix length:",PREFIX_LEN)
print("Prefix trainable params:",prefix_params)
print("Expected:",expected_prefix)
print("Original Phase-1 backbone trainable params:",
      sum(p.numel() for p in backbone.parameters() if p.requires_grad))
assert sum(p.numel() for p in backbone.parameters() if p.requires_grad)==0


Prefix layers: 9
Prefix length: 20
Prefix trainable params: 184320
Expected: 184320
Original Phase-1 backbone trainable params: 0


In [7]:
# ============================================================
# CELL 7 — Download Phase-3 raw JSON files (bypass datasets CastError)
# ============================================================

api = HfApi()
repo_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)

def exact_repo_file(filename):
    matches = [
        f for f in repo_files
        if os.path.basename(f).lower() == filename.lower()
    ]
    assert len(matches) == 1, (filename, matches)
    return matches[0]

FILES = {
    "CL_train": exact_repo_file("CL_train.json"),
    "CL_dev": exact_repo_file("CL_dev.json"),
    "IT_train": exact_repo_file("IT_train.json"),
    "IT_dev": exact_repo_file("IT_dev.json"),
    "CL_test": exact_repo_file("CL_test.json"),
    "IT_test": exact_repo_file("IT_test.json"),
    "label_vocab": exact_repo_file("label_vocab.json"),
}

print(json.dumps(FILES, indent=2))

def load_repo_json(repo_path):
    p = hf_hub_download(
        repo_id=HF_REPO,
        filename=repo_path,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

cl_train_raw = load_repo_json(FILES["CL_train"])
cl_dev_raw   = load_repo_json(FILES["CL_dev"])
it_train_raw = load_repo_json(FILES["IT_train"])
it_dev_raw   = load_repo_json(FILES["IT_dev"])
cl_test_raw  = load_repo_json(FILES["CL_test"])
it_test_raw  = load_repo_json(FILES["IT_test"])
label_vocab_raw = load_repo_json(FILES["label_vocab"])

assert isinstance(cl_train_raw, list) and isinstance(it_train_raw, list)
assert isinstance(cl_dev_raw, list) and isinstance(it_dev_raw, list)
assert isinstance(cl_test_raw, dict) and isinstance(it_test_raw, dict)

print("CL train/dev:", len(cl_train_raw), len(cl_dev_raw))
print("IT train/dev:", len(it_train_raw), len(it_dev_raw))
print("CL/IT test docs:", len(cl_test_raw), len(it_test_raw))


{
  "CL_train": "Phase_3_RR/Train/CL_train.json",
  "CL_dev": "Phase_3_RR/Train/CL_dev.json",
  "IT_train": "Phase_3_RR/Train/IT_train.json",
  "IT_dev": "Phase_3_RR/Train/IT_dev.json",
  "CL_test": "Phase_3_RR/Test/CL_test.json",
  "IT_test": "Phase_3_RR/Test/IT_test.json",
  "label_vocab": "Phase_3_RR/Train/label_vocab.json"
}


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL train/dev: 40 5
IT train/dev: 40 5
CL/IT test docs: 5 5


In [8]:
# ============================================================
# CELL 8 — Labels, examples, class weights, and data checks
# ============================================================

def parse_label_vocab(data):
    if isinstance(data, list):
        labels = list(data)
        label_to_id = {x:i for i,x in enumerate(labels)}
    elif isinstance(data, dict):
        vals = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            label_to_id = {str(k): int(v) for k, v in data.items()}
            assert sorted(label_to_id.values()) == list(range(len(label_to_id)))
            labels = [None] * len(label_to_id)
            for lab, idx in label_to_id.items():
                labels[idx] = lab
        elif all(isinstance(v, str) for v in vals):
            tmp = {int(k): v for k, v in data.items()}
            labels = [tmp[i] for i in sorted(tmp)]
            label_to_id = {x:i for i,x in enumerate(labels)}
        else:
            raise ValueError("Unsupported label_vocab format")
    else:
        raise ValueError("Unsupported label_vocab format")

    return labels, label_to_id

LABELS, LABEL_TO_ID = parse_label_vocab(label_vocab_raw)
ID_TO_LABEL = {v:k for k,v in LABEL_TO_ID.items()}
NUM_LABELS = len(LABELS)

assert NUM_LABELS == 13
print("Labels:")
for i, x in enumerate(LABELS):
    print(i, x)

def normalize_id(doc_id):
    doc_id = str(doc_id)
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id

def validate_labeled_docs(docs, name):
    ids = set()
    n_sent = 0
    for d in docs:
        did = normalize_id(d["id"])
        assert did not in ids, f"{name}: duplicate ID {did}"
        ids.add(did)
        assert isinstance(d["text"], list)
        assert isinstance(d["labels"], list)
        assert len(d["text"]) == len(d["labels"]), did
        assert all(x in LABEL_TO_ID for x in d["labels"]), did
        n_sent += len(d["text"])
    print(f"{name}: docs={len(docs)} sentences={n_sent}")
    return ids

cltr_ids = validate_labeled_docs(cl_train_raw, "CL_train")
cldv_ids = validate_labeled_docs(cl_dev_raw, "CL_dev")
ittr_ids = validate_labeled_docs(it_train_raw, "IT_train")
itdv_ids = validate_labeled_docs(it_dev_raw, "IT_dev")

assert not (cltr_ids & cldv_ids)
assert not (ittr_ids & itdv_ids)
# No document may silently appear in both legal domains/splits after `.txt` normalization.
assert not (cltr_ids & ittr_ids), "CL_train / IT_train ID overlap"
assert not (cltr_ids & itdv_ids), "CL_train / IT_dev ID overlap"
assert not (cldv_ids & ittr_ids), "CL_dev / IT_train ID overlap"
assert not (cldv_ids & itdv_ids), "CL_dev / IT_dev ID overlap"
assert not ((cltr_ids | cldv_ids) & (ittr_ids | itdv_ids)), "Cross-domain labeled ID overlap"

def position_features(i, n):
    if not USE_POS:
        return [0.0, 0.0, 0.0, 0.0]
    return [i / max(1, n - 1), math.log1p(n) / 8.0, float(i == 0), float(i == n - 1)]

# Flatten into sentence-level records while preserving document context.
def make_labeled_records(docs, domain):
    out = []
    for d in docs:
        sents = d["text"]
        labs = d["labels"]
        did = normalize_id(d["id"])
        for i, (sent, lab) in enumerate(zip(sents, labs)):
            out.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
                "label": lab,
                "label_id": LABEL_TO_ID[lab],
            })
    return out

train_records = make_labeled_records(cl_train_raw, "CL") + make_labeled_records(it_train_raw, "IT")
dev_records   = make_labeled_records(cl_dev_raw, "CL") + make_labeled_records(it_dev_raw, "IT")

print("Train sentence records:", len(train_records))
print("Dev sentence records  :", len(dev_records))

train_counts = Counter(x["label"] for x in train_records)
print("\nTrain distribution:")
for lab in LABELS:
    print(f"{lab:30s} {train_counts[lab]:6d}")

# sqrt inverse-frequency; normalize mean to 1; then cap extremes.
N = len(train_records)
C = NUM_LABELS
raw_w = np.array(
    [math.sqrt(N / (C * max(train_counts[lab], 1))) for lab in LABELS],
    dtype=np.float32
)
raw_w = raw_w / raw_w.mean()
class_w = np.clip(raw_w, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX)
class_w = torch.tensor(class_w, dtype=torch.float32, device=DEVICE)

print("\nClass weights:")
for lab, w in zip(LABELS, class_w.detach().cpu().tolist()):
    print(f"{lab:30s} {w:.4f}")


Labels:
0 Fact
1 Issue
2 ArgumentPetitioner
3 ArgumentRespondent
4 Statute
5 Dissent
6 PrecedentReliedUpon
7 PrecedentNotReliedUpon
8 PrecedentOverruled
9 RulingByLowerCourt
10 RatioOfTheDecision
11 RulingByPresentCourt
12 None
CL_train: docs=40 sentences=10593
CL_dev: docs=5 sentences=1396
IT_train: docs=40 sentences=6252
IT_dev: docs=5 sentences=746
Train sentence records: 16845
Dev sentence records  : 2142

Train distribution:
Fact                             4098
Issue                             107
ArgumentPetitioner               1613
ArgumentRespondent               1348
Statute                          1099
Dissent                           284
PrecedentReliedUpon              1945
PrecedentNotReliedUpon            569
PrecedentOverruled                  3
RulingByLowerCourt                874
RatioOfTheDecision               4301
RulingByPresentCourt              307
None                              297

Class weights:
Fact                           0.4500
Issue             

In [9]:
# ============================================================
# CELL 9 — Target-preserving context tokenization
# ============================================================
_ENC_CACHE={}
def enc(text):
    hit=_ENC_CACHE.get(text)
    if hit is None:
        hit=tokenizer.encode(text); _ENC_CACHE[text]=hit
    return hit

def crop_head_tail(ids,budget):
    if len(ids)<=budget: return list(ids)
    if budget<=0: return []
    a=budget//2; return list(ids[:a])+list(ids[-(budget-a):])

def build_context_item(domain,prev_text,target_text,next_text,max_len=MAX_LEN):
    # Markers are ordinary text; no tokenizer-vocabulary changes.
    prefix=enc(f"Domain: {domain}\nPrevious context:\n")
    tmark=enc("\nTarget sentence:\n")
    nmark=enc("\nNext context:\n")
    suffix=enc("\nRhetorical role:\n")
    prev=enc(prev_text)[-PREV_BUDGET:] if prev_text else []
    targ=crop_head_tail(enc(target_text),TARGET_BUDGET)
    nxt=enc(next_text)[:NEXT_BUDGET] if next_text else []

    fixed=len(prefix)+len(tmark)+len(nmark)+len(suffix)
    avail=max_len-fixed
    if avail<16: raise RuntimeError("Marker text unexpectedly consumes MAX_LEN")
    tkeep=min(len(targ), max(16,min(TARGET_BUDGET,avail)))
    remain=avail-tkeep
    pkeep=min(len(prev),remain//2); nkeep=min(len(nxt),remain-pkeep)
    spare=remain-pkeep-nkeep
    add=min(spare,len(prev)-pkeep); pkeep+=add; spare-=add
    add=min(spare,len(nxt)-nkeep); nkeep+=add; spare-=add
    prev=prev[-pkeep:] if pkeep else []; targ=crop_head_tail(targ,tkeep); nxt=nxt[:nkeep] if nkeep else []

    ids=prefix+prev+tmark
    target_start=len(ids)
    ids += targ
    target_end=len(ids)
    ids += nmark+nxt+suffix
    if len(ids)>max_len: raise RuntimeError((len(ids),max_len))
    assert target_end>target_start
    return ids,target_start,target_end

lengths=[]; target_lengths=[]
for r in train_records[:1000]+dev_records[:500]:
    ids,a,b=build_context_item(r['domain'],r['prev'],r['target'],r['next'])
    lengths.append(len(ids)); target_lengths.append(b-a)
print("Context-token sample: mean=",round(float(np.mean(lengths)),2),"p95=",round(float(np.percentile(lengths,95)),2),"max=",max(lengths))
print("Target-token sample : mean=",round(float(np.mean(target_lengths)),2),"p95=",round(float(np.percentile(target_lengths,95)),2))


Context-token sample: mean= 199.64 p95= 278.05 max= 384
Target-token sample : mean= 70.06 p95= 144.0


In [10]:
# ============================================================
# CELL 10 — Dataset, collator, and Model3 Prefix RR classifier
# ============================================================
class RRDataset(Dataset):
    def __init__(self,records,labeled=True):
        assert VOCAB_SIZE<=65535
        self.records=records
        self.labeled=labeled
        self.items=[]
        self.lengths=[]
        for r in records:
            ids,a,b=build_context_item(
                r["domain"],r["prev"],r["target"],r["next"]
            )
            self.items.append((np.asarray(ids,dtype=np.uint16),a,b))
            self.lengths.append(len(ids))
    def __len__(self): return len(self.records)
    def __getitem__(self,idx):
        r=self.records[idx]; ids,a,b=self.items[idx]
        item={
            "input_ids":ids,
            "target_start":a,
            "target_end":b,
            "doc_id":r["doc_id"],
            "sent_idx":r["sent_idx"],
            "domain":r["domain"],
            "pos":r["pos"],
        }
        if self.labeled: item["label_id"]=r["label_id"]
        return item

def collate_rr(batch):
    T=max(len(x["input_ids"]) for x in batch)
    B=len(batch)
    ids=torch.full((B,T),tokenizer.pad_token_id,dtype=torch.long)
    valid=torch.zeros((B,T),dtype=torch.bool)
    tmask=torch.zeros((B,T),dtype=torch.bool)
    pos=torch.tensor([x["pos"] for x in batch],dtype=torch.float32)
    domain_id=torch.tensor(
        [0 if x["domain"]=="CL" else 1 for x in batch],dtype=torch.long
    )
    for i,x in enumerate(batch):
        z=torch.from_numpy(x["input_ids"].astype(np.int64))
        L=len(z); ids[i,:L]=z; valid[i,:L]=True
        tmask[i,x["target_start"]:x["target_end"]]=True
    out={
        "input_ids":ids,"key_valid":valid,"target_mask":tmask,
        "pos":pos,"domain_id":domain_id,
        "doc_id":[x["doc_id"] for x in batch],
        "sent_idx":[x["sent_idx"] for x in batch],
        "domain":[x["domain"] for x in batch],
    }
    if "label_id" in batch[0]:
        out["labels"]=torch.tensor([x["label_id"] for x in batch],dtype=torch.long)
    return out


HEAD_PREFIXES=("head_norm.","pos_mlp.","domain_emb.","fc1.","classifier.")

class RRClassifier(nn.Module):
    def __init__(self,backbone,prefix_module,num_labels=13,dropout=HEAD_DROPOUT):
        super().__init__()
        self.backbone=backbone
        self.prefix_module=prefix_module
        self.head_norm=nn.LayerNorm(2*D_MODEL)
        self.pos_mlp=nn.Sequential(nn.Linear(4,64),nn.GELU())
        self.domain_emb=nn.Embedding(2,32)
        self.fc1=nn.Linear(2*D_MODEL+64+32,HEAD_HIDDEN)
        self.drop=nn.Dropout(dropout)
        self.classifier=nn.Linear(HEAD_HIDDEN,num_labels)

    def forward(self,input_ids,key_valid,target_mask,pos,domain_id):
        prefixes=self.prefix_module()
        h=self.backbone.encode(
            input_ids,key_valid=key_valid,prefix_key_values=prefixes
        ).float()

        vm=key_valid.unsqueeze(-1).to(h.dtype)
        tm=target_mask.unsqueeze(-1).to(h.dtype)
        global_mean=(h*vm).sum(1)/vm.sum(1).clamp(min=1)
        target_mean=(h*tm).sum(1)/tm.sum(1).clamp(min=1)

        x=self.head_norm(torch.cat([target_mean,global_mean],dim=-1))
        z=torch.cat([
            x,self.pos_mlp(pos.float()),self.domain_emb(domain_id)
        ],dim=-1)
        z=F.gelu(self.fc1(self.drop(z)))
        return self.classifier(self.drop(z))


@torch.no_grad()
def init_prefix_from_tokens(m,n_records=PREFIX_INIT_RECORDS,chunk=32,seed=SEED):
    """Scale-match learned Prefix K/V tensors to frozen encoder K/V activations.

    IMPORTANT: this samples TRAIN records only. No dev/test text or labels are used.
    The resulting tensors are still ordinary trainable Prefix-Tuning parameters.
    """
    assert train_records, "train_records is empty"
    pm=m.prefix_module
    P=pm.prefix_len
    nL=pm.n_layers

    rng=np.random.default_rng(seed)
    pick=rng.choice(
        len(train_records),
        size=min(int(n_records),len(train_records)),
        replace=False,
    )
    sampled_records=[train_records[int(i)] for i in pick]
    ds=RRDataset(sampled_records,labeled=False)

    store_k=[[] for _ in range(nL)]
    store_v=[[] for _ in range(nL)]
    cur={}

    # Separate deterministic generators per layer avoid accidental coupling
    # between one layer's sample count and the next layer's initialization.
    layer_gens=[]
    for li in range(nL):
        g=torch.Generator(device="cpu")
        g.manual_seed(seed+1000+li)
        layer_gens.append(g)

    def make_hook(li):
        def hook(mod,inp,out):
            B,T,_=out.shape
            valid=cur["valid"]
            assert valid.shape==(B,T)
            kv=out.view(B,T,3,pm.n_heads,pm.head_dim)[valid]  # [N,3,H,D]
            assert kv.ndim==4 and kv.size(1)==3
            if kv.size(0)==0:
                return
            g=layer_gens[li]
            sel=torch.randperm(kv.size(0),generator=g)[:min(kv.size(0),P*4)]
            sel=sel.to(kv.device)
            store_k[li].append(kv[sel,1].float().cpu())
            store_v[li].append(kv[sel,2].float().cpu())
        return hook

    hooks=[
        m.backbone.encoder[i].attn.qkv.register_forward_hook(make_hook(i))
        for i in range(nL)
    ]

    was_training=m.training
    m.eval()
    try:
        for s in range(0,len(ds),chunk):
            items=[ds[i] for i in range(s,min(s+chunk,len(ds)))]
            batch=collate_rr(items)
            valid=batch["key_valid"].to(DEVICE)
            cur["valid"]=valid
            m.backbone.encode(
                batch["input_ids"].to(DEVICE),
                key_valid=valid,
                prefix_key_values=None,
            )
    finally:
        for h in hooks:
            h.remove()
        m.train(was_training)

    for i in range(nL):
        assert store_k[i] and store_v[i], f"No activation samples collected for layer {i}"
        K=torch.cat(store_k[i],dim=0)
        V=torch.cat(store_v[i],dim=0)
        assert K.size(0)>=P and V.size(0)>=P,(i,K.size(0),V.size(0),P)

        g=layer_gens[i]
        sel=torch.randperm(K.size(0),generator=g)[:P]
        pm.prefix_key[i].copy_(K[sel].to(pm.prefix_key.device))
        pm.prefix_value[i].copy_(V[sel].to(pm.prefix_value.device))

    assert tuple(pm.prefix_key.shape)==(
        N_ENCODER_LAYERS,PREFIX_LEN,N_HEADS,HEAD_DIM
    )
    assert tuple(pm.prefix_value.shape)==tuple(pm.prefix_key.shape)
    assert torch.isfinite(pm.prefix_key).all()
    assert torch.isfinite(pm.prefix_value).all()

    stats={
        "records":len(ds),
        "key_abs_mean":float(pm.prefix_key.abs().mean().item()),
        "value_abs_mean":float(pm.prefix_value.abs().mean().item()),
        "key_std":float(pm.prefix_key.float().std().item()),
        "value_std":float(pm.prefix_value.float().std().item()),
    }
    print(
        "Prefix K/V initialized from frozen TRAIN encoder activations |",
        stats
    )
    return stats


seed_everything(SEED+2)   # same head init in the dev run and in the refit
model=RRClassifier(backbone,prefix_module,NUM_LABELS,HEAD_DROPOUT).to(DEVICE)
PREFIX_INIT_STATS = None
if PREFIX_INIT_FROM_TOKENS:
    PREFIX_INIT_STATS = init_prefix_from_tokens(model)

for n,p in model.named_parameters():
    if n.startswith(HEAD_PREFIXES):
        p.requires_grad=True

# Exact trainability guard.
for n,p in model.named_parameters():
    allowed = n.startswith("prefix_module.") or n.startswith(HEAD_PREFIXES)
    if p.requires_grad:
        assert allowed, f"Unexpected trainable parameter: {n}"
for n,p in model.backbone.named_parameters():
    assert not p.requires_grad,n

trainable=[(n,p.numel()) for n,p in model.named_parameters() if p.requires_grad]
prefix_count=sum(
    p.numel() for n,p in model.named_parameters()
    if p.requires_grad and n.startswith("prefix_module.")
)
head_count=sum(
    p.numel() for n,p in model.named_parameters()
    if p.requires_grad and n.startswith(HEAD_PREFIXES)
)

print("Total params:",f"{sum(p.numel() for p in model.parameters())/1e6:.3f}M")
print("Trainable params:",f"{sum(v for _,v in trainable)/1e6:.3f}M")
print("Trainable %:",f"{100*sum(v for _,v in trainable)/sum(p.numel() for p in model.parameters()):.4f}%")
print("Prefix params:",prefix_count)
print("Task-head params:",head_count)
print("First trainable names:")
for x in trainable[:12]: print(" ",x)


Prefix K/V initialized from frozen TRAIN encoder activations | {'records': 256, 'key_abs_mean': 4.065642833709717, 'value_abs_mean': 1.6514452695846558, 'key_std': 5.262839317321777, 'value_std': 2.3491320610046387}
Total params: 79.415M
Trainable params: 0.622M
Trainable %: 0.7835%
Prefix params: 184320
Task-head params: 437901
First trainable names:
  ('prefix_module.prefix_key', 92160)
  ('prefix_module.prefix_value', 92160)
  ('head_norm.weight', 1024)
  ('head_norm.bias', 1024)
  ('pos_mlp.0.weight', 256)
  ('pos_mlp.0.bias', 64)
  ('domain_emb.weight', 64)
  ('fc1.weight', 430080)
  ('fc1.bias', 384)
  ('classifier.weight', 4992)
  ('classifier.bias', 13)


In [11]:
# ============================================================
# CELL 11 — Dataloaders, optimizer, scheduler
# ============================================================
class SortishBatchSampler(torch.utils.data.Sampler):
    def __init__(self,lengths,batch_size,seed=SEED,bucket_mult=LENGTH_BUCKET_MULT):
        self.lengths=list(map(int,lengths))
        self.batch_size=int(batch_size)
        self.seed=int(seed)
        self.bucket_size=max(self.batch_size,self.batch_size*int(bucket_mult))
        self.epoch=0
    def set_epoch(self,epoch): self.epoch=int(epoch)
    def __len__(self): return math.ceil(len(self.lengths)/self.batch_size)
    def __iter__(self):
        rng=np.random.default_rng(self.seed+self.epoch)
        idx=np.arange(len(self.lengths),dtype=np.int64); rng.shuffle(idx)
        batches=[]
        for s in range(0,len(idx),self.bucket_size):
            chunk=idx[s:s+self.bucket_size]
            chunk=sorted(chunk.tolist(),key=lambda i:self.lengths[i],reverse=True)
            batches.extend(
                chunk[j:j+self.batch_size]
                for j in range(0,len(chunk),self.batch_size)
            )
        rng.shuffle(batches)
        yield from batches

def make_loader(records,labeled,batch_size,shuffle):
    t0=time.time(); ds=RRDataset(records,labeled)
    print(f"tokenized {len(ds)} records in {time.time()-t0:.1f}s")
    kw=dict(
        collate_fn=collate_rr,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type=="cuda"),
        persistent_workers=NUM_WORKERS>0,
    )
    if shuffle and LENGTH_GROUPED_TRAINING:
        return DataLoader(
            ds,batch_sampler=SortishBatchSampler(ds.lengths,batch_size,seed=SEED),**kw
        )
    if shuffle:
        g=torch.Generator(); g.manual_seed(SEED)
        return DataLoader(ds,batch_size=batch_size,shuffle=True,generator=g,**kw)
    order=sorted(range(len(ds)),key=lambda i:ds.lengths[i])
    return DataLoader(Subset(ds,order),batch_size=batch_size,shuffle=False,**kw)

def make_adamw(prefix_params,head_params):
    groups=[
        {"params":prefix_params,"lr":PREFIX_LR,"weight_decay":0.0},
        {"params":head_params,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
    ]
    if DEVICE.type=="cuda":
        try:
            opt=torch.optim.AdamW(groups,fused=True)
            print("AdamW fused: True"); return opt
        except Exception as e:
            print("Fused AdamW unavailable; standard AdamW:",repr(e)[:120])
    print("AdamW fused: False")
    return torch.optim.AdamW(groups)

train_loader=make_loader(train_records,True,MICRO_BATCH_SIZE,True)
dev_loader=make_loader(dev_records,True,EVAL_BATCH_SIZE,False)

prefix_params=[]; head_params=[]
for n,p in model.named_parameters():
    if not p.requires_grad: continue
    if n.startswith("prefix_module."): prefix_params.append(p)
    else:
        assert n.startswith(HEAD_PREFIXES),n
        head_params.append(p)
assert prefix_params and head_params

optimizer=make_adamw(prefix_params,head_params)
updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=NUM_EPOCHS*updates_per_epoch
warmup_updates=max(1,int(total_updates*WARMUP_RATIO))

def lr_mult(step):
    if step<warmup_updates:
        return max(1e-8,step/warmup_updates)
    p=(step-warmup_updates)/max(1,total_updates-warmup_updates)
    return 0.5*(1+math.cos(math.pi*min(max(p,0.0),1.0)))

scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lr_mult)
criterion=nn.CrossEntropyLoss(weight=class_w,label_smoothing=LABEL_SMOOTHING)
amp_enabled=bool(USE_AMP and DEVICE.type=="cuda")
scaler=torch.amp.GradScaler("cuda",enabled=amp_enabled)

print(
    "Train batches/epoch:",len(train_loader),
    "updates/epoch:",updates_per_epoch,
    "total updates:",total_updates,
    "warmup:",warmup_updates,
    "AMP:",amp_enabled,
)


tokenized 16845 records in 1.6s
tokenized 2142 records in 0.1s
AdamW fused: True
Train batches/epoch: 2106 updates/epoch: 1053 total updates: 6318 warmup: 379 AMP: True


In [12]:
# ============================================================
# CELL 12 — Metrics and evaluation
# ============================================================
def _forward_batch(m,batch):
    return m(batch['input_ids'].to(DEVICE,non_blocking=True),batch['key_valid'].to(DEVICE,non_blocking=True),
             batch['target_mask'].to(DEVICE,non_blocking=True),batch['pos'].to(DEVICE,non_blocking=True),batch['domain_id'].to(DEVICE,non_blocking=True))

@torch.no_grad()
def predict_loader(m,loader):
    m.eval(); yt=[]; yp=[]; meta=[]
    for batch in loader:
        with torch.amp.autocast(device_type='cuda',dtype=torch.float16,enabled=amp_enabled): logits=_forward_batch(m,batch)
        yp.extend(logits.float().argmax(-1).cpu().tolist())
        if 'labels' in batch: yt.extend(batch['labels'].tolist())
        meta.extend(zip(batch['doc_id'],batch['sent_idx'],batch['domain']))
    return yt,yp,meta

def score_ids(y_true,y_pred,title='Combined'):
    ids=list(range(NUM_LABELS)); f=f1_score(y_true,y_pred,labels=ids,average='macro',zero_division=0); p=precision_score(y_true,y_pred,labels=ids,average='macro',zero_division=0); r=recall_score(y_true,y_pred,labels=ids,average='macro',zero_division=0)
    print(f"{title}: F1={f:.6f} P={p:.6f} R={r:.6f}"); return f,p,r

def print_dev_details(yt,yp):
    present=sorted(set(yt)); absent=[i for i in range(NUM_LABELS) if i not in present]
    print(f"Dev contains {len(present)}/{NUM_LABELS} labels; formal Macro-F1 ceiling from label absence = {len(present)/NUM_LABELS:.6f}")
    if absent: print('Absent dev labels:',[LABELS[i] for i in absent])
    pc,gc=Counter(yp),Counter(yt); fp=f1_score(yt,yp,labels=present,average='macro',zero_division=0)
    print(f"   Macro-F1 over {len(present)} labels present in dev: {fp:.6f}")
    print('   gold counts:',{LABELS[k]:gc[k] for k in present}); print('   pred counts:',{LABELS[k]:pc[k] for k in sorted(pc)})
    print("\nPer-label report (all 13 labels explicitly included):")
    print(classification_report(yt,yp,labels=list(range(NUM_LABELS)),target_names=LABELS,digits=4,zero_division=0))

def detailed_dev_eval(m,verbose=True):
    yt,yp,meta=predict_loader(m,dev_loader); combined=score_ids(yt,yp,'Combined CL+IT')
    _pc0=Counter(yp)
    if max(_pc0.values())/max(1,len(yp))>0.60: print('   ⚠️ >60% of predictions are one class — possible collapse:',{LABELS[k]:v for k,v in _pc0.items()})
    for dom in ['CL','IT']:
        idx=[i for i,x in enumerate(meta) if x[2]==dom]; score_ids([yt[i] for i in idx],[yp[i] for i in idx],dom)
    if verbose: print_dev_details(yt,yp)
    return combined,yt,yp,meta

@torch.no_grad()
def predict_logits(m,loader):
    m.eval(); outs=[]; meta=[]; yt=[]
    for batch in loader:
        with torch.amp.autocast(device_type='cuda',dtype=torch.float16,enabled=amp_enabled): lg=_forward_batch(m,batch)
        outs.append(lg.float().cpu().numpy()); meta.extend(zip(batch['doc_id'],batch['sent_idx'],batch['domain']))
        if 'labels' in batch: yt.extend(batch['labels'].tolist())
    return np.concatenate(outs,0),meta,yt


In [13]:
# ============================================================
# CELL 13 — Train Prefix Tuning; select by combined Macro-F1
# ============================================================
def fmt_sec(s):
    s=max(0,int(s)); h,s=divmod(s,3600); m,s=divmod(s,60)
    return f"{h:02d}:{m:02d}:{s:02d}"

best_f1=-1.0
best_epoch=None
best_frac=None
best_batch_in_epoch=None
best_global_update=None
bad_evals=0
global_update=0
history=[]
train_start=time.time()
optimizer.zero_grad(set_to_none=True)

for epoch in range(1,NUM_EPOCHS+1):
    if hasattr(train_loader.batch_sampler,"set_epoch"):
        train_loader.batch_sampler.set_epoch(epoch)

    model.train()
    running=torch.zeros((),device=DEVICE,dtype=torch.float32)
    n_batches=0
    ep0=time.time()

    # Two deterministic eval points by default: half epoch and full epoch.
    eval_steps=sorted(set(
        min(len(train_loader),max(1,int(round(len(train_loader)*j/EVAL_PER_EPOCH))))
        for j in range(1,EVAL_PER_EPOCH+1)
    ))
    if epoch==1:
        print("Dev evaluation at batches:",eval_steps,"of",len(train_loader))

    stop_training=False
    for step,batch in enumerate(train_loader,1):
        labels=batch["labels"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast(
            device_type="cuda",dtype=torch.float16,enabled=amp_enabled
        ):
            logits=_forward_batch(model,batch)
            loss=criterion(logits,labels)
            scaled=loss/GRAD_ACCUM_STEPS

        if step%NAN_CHECK_EVERY==0 or step==len(train_loader):
            if not bool(torch.isfinite(loss.detach()).item()):
                raise FloatingPointError(f"Non-finite loss epoch={epoch} batch={step}")

        scaler.scale(scaled).backward()
        running.add_(loss.detach().float()); n_batches+=1

        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],GRAD_CLIP
            )
            old_scale=scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            new_scale=scaler.get_scale()
            optimizer.zero_grad(set_to_none=True)
            if new_scale>=old_scale:
                scheduler.step(); global_update+=1

        if step%PRINT_EVERY==0 or step==len(train_loader):
            elapsed=time.time()-ep0
            rate=elapsed/max(step,1)
            eta_ep=rate*(len(train_loader)-step)
            completed=(epoch-1)+step/len(train_loader)
            eta_total=rate*len(train_loader)*max(0,NUM_EPOCHS-completed)
            print(
                f"epoch={epoch} batch={step}/{len(train_loader)} "
                f"loss={float((running/max(n_batches,1)).item()):.4f} "
                f"lr_prefix={optimizer.param_groups[0]['lr']:.2e} "
                f"lr_head={optimizer.param_groups[1]['lr']:.2e} "
                f"elapsed={fmt_sec(elapsed)} ETA_epoch={fmt_sec(eta_ep)} "
                f"ETA_maxtrain≈{fmt_sec(eta_total)}",
                flush=True
            )

        if step in eval_steps:
            metrics,yt,yp,meta=detailed_dev_eval(model,verbose=VERBOSE_EVAL_REPORT)
            mf,mp,mr=metrics
            frac=(epoch-1)+step/len(train_loader)

            counts=Counter(yp)
            max_share=max(counts.values())/max(1,len(yp))
            if max_share>0.65:
                print(f"⚠️ Collapse warning: dominant predicted class share={max_share:.3f}")

            row={
                "epoch":epoch,"epoch_frac":float(frac),
                "batch_in_epoch":int(step),
                "global_update":int(global_update),
                "train_loss":float((running/max(n_batches,1)).item()),
                "macro_f1":mf,"macro_precision":mp,"macro_recall":mr,
                "elapsed_sec":time.time()-ep0,
            }
            history.append(row)
            print("Eval summary:",row)

            if mf>best_f1:
                best_f1=mf
                best_epoch=int(math.ceil(frac))
                best_frac=float(frac)
                best_batch_in_epoch=int(step)
                best_global_update=int(global_update)
                bad_evals=0

                task_state={
                    k:v.detach().cpu()
                    for k,v in model.state_dict().items()
                    if k.startswith("prefix_module.") or k.startswith(HEAD_PREFIXES)
                }
                torch.save({
                    "task_state":task_state,
                    "best_epoch":best_epoch,
                    "best_frac":best_frac,
                    "best_batch_in_epoch":best_batch_in_epoch,
                    "best_global_update":best_global_update,
                    "best_macro_f1":best_f1,
                    "history":history,
                    "labels":LABELS,
                    "label_to_id":LABEL_TO_ID,
                    "config":{
                        "experiment":"Phase3 Exp4 Model3 Prefix Tuning",
                        "prefix_type":"encoder_self_attention_KV",
                        "prefix_len":PREFIX_LEN,
                        "prefix_dropout":PREFIX_DROPOUT,
                        "prefix_lr":PREFIX_LR,
                        "head_lr":HEAD_LR,
                        "max_len":MAX_LEN,
                        "context_sents":CONTEXT_SENTS,
                    },
                },BEST_CKPT)
                print(
                    f"✅ NEW BEST at epoch {frac:.3f} | batch={step} | "
                    f"update={global_update}: {BEST_CKPT}"
                )
                if not VERBOSE_EVAL_REPORT: print_dev_details(yt,yp)
            else:
                bad_evals+=1

            print(
                f"Best so far: epoch_frac={best_frac:.3f} | "
                f"batch={best_batch_in_epoch} | update={best_global_update} | "
                f"Macro-F1={best_f1:.6f} | non-improving evals={bad_evals}\n"
            )

            if bad_evals>=EARLY_STOP_PATIENCE_EVALS:
                print(
                    f"⏹️ Early stopping after {bad_evals} non-improving dev evaluations."
                )
                stop_training=True
                break

            model.train()

    if stop_training:
        break

with open(ROOT/"history.json","w",encoding="utf-8") as f:
    json.dump(history,f,indent=2)

assert best_frac is not None and best_batch_in_epoch is not None
print("Saved:",ROOT/"history.json")
print("Training complete | elapsed:",fmt_sec(time.time()-train_start))
print(
    "BEST_EPOCH =",best_epoch,
    "| BEST_EPOCH_FRAC =",best_frac,
    "| BEST_BATCH =",best_batch_in_epoch,
    "| BEST_UPDATE =",best_global_update,
    "| BEST_MACRO_F1 =",best_f1
)


Dev evaluation at batches: [1053, 2106] of 2106
epoch=1 batch=250/2106 loss=2.3180 lr_prefix=9.89e-05 lr_head=2.31e-04 elapsed=00:00:08 ETA_epoch=00:01:00 ETA_maxtrain≈00:06:45
epoch=1 batch=500/2106 loss=2.2305 lr_prefix=1.98e-04 lr_head=4.62e-04 elapsed=00:00:14 ETA_epoch=00:00:46 ETA_maxtrain≈00:05:52
epoch=1 batch=750/2106 loss=2.1814 lr_prefix=2.96e-04 lr_head=6.91e-04 elapsed=00:00:21 ETA_epoch=00:00:37 ETA_maxtrain≈00:05:32
epoch=1 batch=1000/2106 loss=2.1314 lr_prefix=3.00e-04 lr_head=6.99e-04 elapsed=00:00:27 ETA_epoch=00:00:30 ETA_maxtrain≈00:05:19
Combined CL+IT: F1=0.108284 P=0.136551 R=0.144224
CL: F1=0.103924 P=0.168038 R=0.138284
IT: F1=0.112463 P=0.114633 R=0.150078
Eval summary: {'epoch': 1, 'epoch_frac': 0.5, 'batch_in_epoch': 1053, 'global_update': 525, 'train_loss': 2.1207358837127686, 'macro_f1': 0.10828442883622678, 'macro_precision': 0.1365512285885586, 'macro_recall': 0.14422411300657134, 'elapsed_sec': 31.35372567176819}
✅ NEW BEST at epoch 0.500 | batch=1053 |

In [14]:
# ============================================================
# CELL 14 — Reload the exact best Prefix checkpoint
# ============================================================
best_obj=torch.load(BEST_CKPT,map_location="cpu",weights_only=False)
missing,unexpected=model.load_state_dict(best_obj["task_state"],strict=False)

# Only frozen Phase-1 keys should be missing because task_state intentionally stores
# only Prefix + task-head weights.
bad_missing=[
    k for k in missing
    if k.startswith("prefix_module.") or k.startswith(HEAD_PREFIXES)
]
assert not bad_missing,bad_missing
assert not unexpected,unexpected

best_f1=float(best_obj["best_macro_f1"])
best_epoch=int(best_obj["best_epoch"])
best_frac=float(best_obj.get("best_frac",best_epoch))
print(
    "Reloaded best Prefix checkpoint:",
    "epoch_frac=",best_frac,
    "Macro-F1=",best_f1
)

# ---- helper used by every later cell: (meta, label ids) -> {doc_id: [label strings in sentence order]}
def records_to_prediction(meta,ids):
    by_doc={}
    for (did,si,dom),y in zip(meta,ids):
        by_doc.setdefault(did,[]).append((int(si),int(y)))
    out={}
    for did,items in by_doc.items():
        items.sort()
        assert [s for s,_ in items]==list(range(len(items))),f"non-contiguous sentence indices in {did}"
        out[did]=[LABELS[y] for _,y in items]
    return out

# ---- reference files for the exact local evaluator (Cells 16-17)
def write_truth(docs,path):
    with open(path,"w",encoding="utf-8") as f:
        json.dump({normalize_id(d["id"]):d["labels"] for d in docs},f,ensure_ascii=False)
write_truth(cl_dev_raw,CL_DEV_TRUTH); write_truth(it_dev_raw,IT_DEV_TRUTH)
with open(LABELS_JSON,"w",encoding="utf-8") as f: json.dump(LABELS,f,ensure_ascii=False)

# ---- re-score the reloaded checkpoint and write the DEV prediction file
(rf,rp,rr),yt_chk,yp_chk,meta_chk=detailed_dev_eval(model,verbose=True)
assert abs(rf-best_f1)<5e-3,f"Reloaded checkpoint F1 {rf:.6f} != recorded best {best_f1:.6f}"
dev_pred=records_to_prediction(meta_chk,yp_chk)
for d in cl_dev_raw+it_dev_raw: dev_pred.setdefault(normalize_id(d["id"]),[])
with open(DEV_PRED,"w",encoding="utf-8") as f: json.dump(dev_pred,f,ensure_ascii=False,indent=2)
print("Saved:",DEV_PRED,"| docs:",len(dev_pred))

# Evaluator artifact preflight — these are required by Cells 16–17.
for _p in [CL_DEV_TRUTH,IT_DEV_TRUTH,LABELS_JSON,DEV_PRED]:
    assert Path(_p).is_file(),f"Missing evaluator artifact: {_p}"
print("✅ Evaluator artifact preflight passed.")



Reloaded best Prefix checkpoint: epoch_frac= 3.0 Macro-F1= 0.18172548633282323
Combined CL+IT: F1=0.181725 P=0.227161 R=0.198538
CL: F1=0.163900 P=0.267264 R=0.176647
IT: F1=0.163865 P=0.187486 R=0.190185
Dev contains 11/13 labels; formal Macro-F1 ceiling from label absence = 0.846154
Absent dev labels: ['Dissent', 'PrecedentOverruled']
   Macro-F1 over 11 labels present in dev: 0.214766
   gold counts: {'Fact': 444, 'Issue': 15, 'ArgumentPetitioner': 400, 'ArgumentRespondent': 272, 'Statute': 120, 'PrecedentReliedUpon': 254, 'PrecedentNotReliedUpon': 50, 'RulingByLowerCourt': 102, 'RatioOfTheDecision': 403, 'RulingByPresentCourt': 49, 'None': 33}
   pred counts: {'Fact': 674, 'ArgumentPetitioner': 17, 'ArgumentRespondent': 199, 'Statute': 120, 'Dissent': 21, 'PrecedentReliedUpon': 99, 'PrecedentNotReliedUpon': 22, 'RatioOfTheDecision': 945, 'RulingByPresentCourt': 22, 'None': 23}

Per-label report (all 13 labels explicitly included):
                        precision    recall  f1-sco

In [15]:
# ============================================================
# CELL 15 — EXACT local Phase-3 evaluator script supplied by course
# ============================================================

EVALUATOR_CODE = r'''#!/usr/bin/env python3
"""Local evaluator for Phase 3.

The competition scorer evaluates two gold/reference JSON files together and
one participant prediction.json. This local evaluator mirrors that setup
using the student's own held-out splits.
"""
import argparse
import json
import sys

from sklearn.metrics import f1_score, precision_score, recall_score

EXPECTED_LABEL_COUNT = 13


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def normalize_id(doc_id):
    if not isinstance(doc_id, str) or not doc_id.strip():
        raise ValueError("Document IDs must be non-empty strings.")
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id


def load_vocab(data):
    if isinstance(data, list):
        labels = data
    elif isinstance(data, dict) and data:
        values = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in values):
            labels = list(data.keys())
        elif all(isinstance(v, str) for v in values):
            labels = values
        else:
            raise ValueError("Unsupported labels.json format.")
    else:
        raise ValueError("labels.json must be a non-empty list or object.")

    if len(labels) != EXPECTED_LABEL_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_LABEL_COUNT} labels, found {len(labels)}."
        )
    if any(not isinstance(label, str) or not label.strip() for label in labels):
        raise ValueError("Labels must be unique non-empty strings.")
    if len(set(labels)) != len(labels):
        raise ValueError("Labels must be unique non-empty strings.")
    return labels


def normalize_reference(data, source_name):
    if not isinstance(data, dict):
        raise ValueError(f"{source_name}: reference data must be a JSON object.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"{source_name}: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def normalize_predictions(data):
    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object mapping document IDs to label lists.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"prediction.json: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate Phase 3 sentence-label predictions on two local held-out "
            "reference splits, mirroring the competition's combined scoring setup."
        )
    )
    parser.add_argument("--truth-cl", required=True)
    parser.add_argument("--truth-it", required=True)
    parser.add_argument("--prediction", required=True)
    parser.add_argument("--labels", required=True)
    args = parser.parse_args()

    truth_cl = normalize_reference(load_json(args.truth_cl), "--truth-cl")
    truth_it = normalize_reference(load_json(args.truth_it), "--truth-it")
    predictions = normalize_predictions(load_json(args.prediction))
    labels = load_vocab(load_json(args.labels))
    allowed = set(labels)

    overlap = set(truth_cl) & set(truth_it)
    if overlap:
        sample = sorted(overlap)[:5]
        suffix = "..." if len(overlap) > 5 else ""
        raise ValueError(
            "The two reference splits contain overlapping document IDs: "
            + ", ".join(sample)
            + suffix
        )

    truth = {}
    truth.update(truth_cl)
    truth.update(truth_it)

    if set(truth) != set(predictions):
        missing = sorted(set(truth) - set(predictions))
        extra = sorted(set(predictions) - set(truth))
        parts = [f"missing={len(missing)}", f"extra={len(extra)}"]
        if missing:
            parts.append(f"example missing ID(s)={missing[:5]}")
        if extra:
            parts.append(f"example extra ID(s)={extra[:5]}")
        raise ValueError("ID mismatch: " + "; ".join(parts))

    y_true = []
    y_pred = []

    for doc_id in truth:
        gold = truth[doc_id]
        pred = predictions[doc_id]

        if not isinstance(gold, list) or not isinstance(pred, list):
            raise ValueError(f"{doc_id}: labels must be JSON lists.")
        if len(gold) != len(pred):
            raise ValueError(
                f"{doc_id}: sentence-count mismatch ({len(gold)} vs {len(pred)})."
            )
        if any(label not in allowed for label in gold + pred):
            raise ValueError(f"{doc_id}: unknown label encountered.")

        y_true.extend(gold)
        y_pred.extend(pred)

    score = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"CL documents: {len(truth_cl)}")
    print(f"IT documents: {len(truth_it)}")
    print(f"Total documents: {len(truth)}")
    print(f"Total sentences: {len(y_true)}")
    print(f"Macro-F1:        {score:.6f}")
    print(f"Macro-Precision: {macro_precision:.6f}")
    print(f"Macro-Recall:    {macro_recall:.6f}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"EVALUATION FAILED: {exc}", file=sys.stderr)
        sys.exit(1)
'''

with open(EVAL_SCRIPT, "w", encoding="utf-8") as f:
    f.write(EVALUATOR_CODE)

print("Saved exact evaluator to:", EVAL_SCRIPT)


Saved exact evaluator to: /kaggle/working/phase3_exp4_model3_prefix/evaluate_phase3.py


In [16]:
# ============================================================
# CELL 16 — RUN THE OFFICIAL-STYLE LOCAL EVALUATOR
# Do not continue to final refit until this passes.
# ============================================================

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--truth-cl", str(CL_DEV_TRUTH),
    "--truth-it", str(IT_DEV_TRUTH),
    "--prediction", str(DEV_PRED),
    "--labels", str(LABELS_JSON),
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("\nSTDOUT:\n", result.stdout)
if result.stderr:
    print("\nSTDERR:\n", result.stderr)

assert result.returncode == 0, "❌ Phase-3 evaluator failed."
print("✅ DEV prediction passed the exact Phase-3 evaluator.")


Running:
/usr/bin/python3 /kaggle/working/phase3_exp4_model3_prefix/evaluate_phase3.py --truth-cl /kaggle/working/phase3_exp4_model3_prefix/cl_dev_truth.json --truth-it /kaggle/working/phase3_exp4_model3_prefix/it_dev_truth.json --prediction /kaggle/working/phase3_exp4_model3_prefix/prediction_dev.json --labels /kaggle/working/phase3_exp4_model3_prefix/labels.json

STDOUT:
 CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.181725
Macro-Precision: 0.227161
Macro-Recall:    0.198538

✅ DEV prediction passed the exact Phase-3 evaluator.


In [17]:
# ============================================================
# CELL 17 — Optional transition-aware decoding tuned on DEV
#
# Improvements:
#   * CL and IT transition/prior statistics estimated separately
#   * additive smoothing
#   * no hard "impossible label" bans
#   * only adopt decoding when DEV Macro-F1 improves by >= 0.005
# ============================================================

from sklearn.metrics import f1_score as _f1

def transition_matrix(docs_list, smooth=1.0):
    T = np.full((NUM_LABELS, NUM_LABELS), float(smooth), dtype=np.float64)
    for d in docs_list:
        ids = [LABEL_TO_ID[x] for x in d["labels"]]
        for a, b in zip(ids[:-1], ids[1:]):
            T[a, b] += 1.0
    T /= T.sum(1, keepdims=True)
    return np.log(T)

def class_log_prior(docs_list, smooth=1.0):
    c = np.full(NUM_LABELS, float(smooth), dtype=np.float64)
    for d in docs_list:
        for x in d["labels"]:
            c[LABEL_TO_ID[x]] += 1.0
    return np.log(c / c.sum())

def log_softmax_np(x):
    x = x - x.max(-1, keepdims=True)
    return x - np.log(np.exp(x).sum(-1, keepdims=True))

def viterbi_decode(logp, trans, beta):
    n, K = logp.shape
    score = logp[0].copy()
    back = np.zeros((n, K), dtype=np.int32)
    tr = beta * trans

    for t in range(1, n):
        cand = score[:, None] + tr
        back[t] = cand.argmax(0)
        score = cand[back[t], np.arange(K)] + logp[t]

    path = np.zeros(n, dtype=np.int64)
    path[-1] = int(score.argmax())

    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]

    return path

def build_domain_stats(cl_docs, it_docs):
    return {
        "CL": {
            "trans": transition_matrix(cl_docs, smooth=1.0),
            "prior": class_log_prior(cl_docs, smooth=1.0),
        },
        "IT": {
            "trans": transition_matrix(it_docs, smooth=1.0),
            "prior": class_log_prior(it_docs, smooth=1.0),
        },
    }

def decode_docs(logits, meta, domain_stats, alpha, beta):
    # logits aligned with meta=(doc_id, sent_idx, domain)
    by_doc = {}
    for i, (did, si, dom) in enumerate(meta):
        by_doc.setdefault(did, []).append((si, i, dom))

    pred = np.zeros(len(meta), dtype=np.int64)

    for did, items in by_doc.items():
        items.sort(key=lambda x: x[0])
        idx = [i for _, i, _ in items]
        dom = items[0][2]
        stats = domain_stats[dom]

        lp = (
            log_softmax_np(logits[idx].astype(np.float64))
            - alpha * stats["prior"][None, :]
        )

        pred[idx] = (
            viterbi_decode(lp, stats["trans"], beta)
            if beta > 0
            else lp.argmax(-1)
        )

    return pred

# Tune only on held-out DEV predictions.
dev_logits, dev_meta, dev_y = predict_logits(model, dev_loader)
np.savez_compressed(
    ROOT / "dev_logits.npz",
    logits=dev_logits,
    y=np.asarray(dev_y, dtype=np.int64),
    doc_id=np.asarray([x[0] for x in dev_meta]),
    sent_idx=np.asarray([x[1] for x in dev_meta], dtype=np.int64),
    domain=np.asarray([x[2] for x in dev_meta]),
)

domain_stats_train = build_domain_stats(
    cl_train_raw,
    it_train_raw,
)

def dev_f1(alpha, beta):
    p = decode_docs(
        dev_logits,
        dev_meta,
        domain_stats_train,
        alpha,
        beta
    )
    return _f1(
        dev_y,
        p,
        labels=list(range(NUM_LABELS)),
        average="macro",
        zero_division=0
    )

rows = []
for alpha in [-0.25, 0.0, 0.25, 0.5, 0.75]:
    for beta in [0.0, 0.25, 0.5, 1.0]:
        rows.append((dev_f1(alpha, beta), alpha, beta))

base_f1 = dev_f1(0.0, 0.0)
rows.sort(key=lambda r: (-r[0], r[2], abs(r[1])))

print(f"Baseline argmax (alpha=0, beta=0): {base_f1:.6f}")
print("Top decoding settings (dev F1, alpha, beta):")
for r in rows[:8]:
    print(f"  {r[0]:.6f}  alpha={r[1]:+.2f}  beta={r[2]:.2f}")

if rows[0][0] - base_f1 >= 0.005:
    DECODE_ALPHA, DECODE_BETA = rows[0][1], rows[0][2]
else:
    DECODE_ALPHA, DECODE_BETA = 0.0, 0.0

print("SELECTED decoding: alpha=", DECODE_ALPHA, "beta=", DECODE_BETA)

dec_ids = decode_docs(
    dev_logits,
    dev_meta,
    domain_stats_train,
    DECODE_ALPHA,
    DECODE_BETA
)

dev_pred_dec = records_to_prediction(dev_meta, dec_ids)
DEV_PRED_DEC = ROOT / "prediction_dev_decoded.json"

with open(DEV_PRED_DEC, "w", encoding="utf-8") as f:
    json.dump(dev_pred_dec, f, ensure_ascii=False, indent=2)

# Run the exact supplied evaluator again on the decoded DEV prediction.
res = subprocess.run(
    [
        sys.executable,
        str(EVAL_SCRIPT),
        "--truth-cl", str(CL_DEV_TRUTH),
        "--truth-it", str(IT_DEV_TRUTH),
        "--prediction", str(DEV_PRED_DEC),
        "--labels", str(LABELS_JSON),
    ],
    capture_output=True,
    text=True
)

print(res.stdout)
if res.stderr:
    print(res.stderr)

assert res.returncode == 0, "Decoded DEV prediction failed the exact evaluator."
print("✅ Decoded DEV prediction passed the exact Phase-3 evaluator.")


Baseline argmax (alpha=0, beta=0): 0.181725
Top decoding settings (dev F1, alpha, beta):
  0.218114  alpha=+0.50  beta=0.25
  0.212313  alpha=+0.50  beta=0.50
  0.202638  alpha=+0.25  beta=0.25
  0.201182  alpha=+0.25  beta=0.50
  0.200463  alpha=+0.25  beta=0.00
  0.196324  alpha=+0.50  beta=1.00
  0.195323  alpha=+0.75  beta=1.00
  0.187655  alpha=+0.25  beta=1.00
SELECTED decoding: alpha= 0.5 beta= 0.25
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.218114
Macro-Precision: 0.290374
Macro-Recall:    0.237867

✅ Decoded DEV prediction passed the exact Phase-3 evaluator.


In [18]:
# ============================================================
# CELL 18 — Fresh Phase-1 Model3 + Prefix model for train+dev refit
# ============================================================
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

BEST_EPOCH=int(best_obj["best_epoch"])
BEST_FRAC=float(best_obj.get("best_frac",BEST_EPOCH))
BEST_BATCH=int(best_obj.get("best_batch_in_epoch",0))
BEST_UPDATE=int(best_obj.get("best_global_update",0))
assert BEST_EPOCH>=1 and 0.0<BEST_FRAC<=NUM_EPOCHS

print(
    "Final refit target selected from dev:",
    f"epoch_frac={BEST_FRAC:.3f}",
    f"| selected_batch={BEST_BATCH}",
    f"| selected_update={BEST_UPDATE}",
)

all_labeled_records=(
    make_labeled_records(cl_train_raw,"CL")
    +make_labeled_records(it_train_raw,"IT")
    +make_labeled_records(cl_dev_raw,"CL")
    +make_labeled_records(it_dev_raw,"IT")
)

all_counts=Counter(x["label"] for x in all_labeled_records)
N_all=len(all_labeled_records)
raw=np.array([
    math.sqrt(N_all/(NUM_LABELS*max(all_counts[lab],1)))
    for lab in LABELS
],dtype=np.float32)
raw/=raw.mean()
class_w_all=torch.tensor(
    np.clip(raw,CLASS_WEIGHT_MIN,CLASS_WEIGHT_MAX),
    dtype=torch.float32,device=DEVICE
)

def reset_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def build_fresh_rr_from_phase1():
    reset_seed(SEED)
    bb=build_backbone()
    o=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
    st=o["model"] if isinstance(o,dict) and "model" in o else o
    bb.load_state_dict(st,strict=True)
    for p in bb.parameters(): p.requires_grad=False

    reset_seed(SEED+1)   # identical to Cell 6
    pm=EncoderKVPrefix(
        N_ENCODER_LAYERS,PREFIX_LEN,N_HEADS,HEAD_DIM
    )
    reset_seed(SEED+2)   # identical to Cell 10
    m=RRClassifier(bb,pm,NUM_LABELS,HEAD_DROPOUT)

    for n,p in m.named_parameters():
        if n.startswith(HEAD_PREFIXES): p.requires_grad=True
    for n,p in m.backbone.named_parameters():
        assert not p.requires_grad,n
    m=m.to(DEVICE)
    init_stats=None
    if PREFIX_INIT_FROM_TOKENS:
        init_stats=init_prefix_from_tokens(m)   # same TRAIN-only initialization as heldout run
    m._prefix_init_stats=init_stats
    return m

final_model=build_fresh_rr_from_phase1()
all_loader=make_loader(all_labeled_records,True,MICRO_BATCH_SIZE,True)

final_prefix=[]; final_head=[]
for n,p in final_model.named_parameters():
    if not p.requires_grad: continue
    if n.startswith("prefix_module."): final_prefix.append(p)
    else:
        assert n.startswith(HEAD_PREFIXES),n
        final_head.append(p)

final_optimizer=make_adamw(final_prefix,final_head)
final_updates_per_epoch=math.ceil(len(all_loader)/GRAD_ACCUM_STEPS)

REFIT_TOTAL_BATCHES=int(
    round(BEST_FRAC*len(all_loader)/GRAD_ACCUM_STEPS)
)*GRAD_ACCUM_STEPS
REFIT_TOTAL_BATCHES=max(GRAD_ACCUM_STEPS,REFIT_TOTAL_BATCHES)

REFIT_FULL_EPOCHS=REFIT_TOTAL_BATCHES//len(all_loader)
REFIT_REMAINDER_BATCHES=REFIT_TOTAL_BATCHES%len(all_loader)
print(
    "Refit target:",REFIT_TOTAL_BATCHES,
    "micro-batches total | loader batches/epoch:",len(all_loader),
    "| full epochs:",REFIT_FULL_EPOCHS,
    "| remainder batches:",REFIT_REMAINDER_BATCHES
)

SCHED_EPOCHS=NUM_EPOCHS if REFIT_MATCH_ORIGINAL_SCHEDULE else max(1,math.ceil(BEST_FRAC))
final_total_updates=SCHED_EPOCHS*final_updates_per_epoch
final_warmup=max(1,int(final_total_updates*WARMUP_RATIO))

def final_lr_mult(step):
    if step<final_warmup:
        return max(1e-8,step/final_warmup)
    p=(step-final_warmup)/max(1,final_total_updates-final_warmup)
    return 0.5*(1+math.cos(math.pi*min(max(p,0.0),1.0)))

final_scheduler=torch.optim.lr_scheduler.LambdaLR(
    final_optimizer,lr_lambda=final_lr_mult
)
final_criterion=nn.CrossEntropyLoss(
    weight=class_w_all,label_smoothing=LABEL_SMOOTHING
)
final_scaler=torch.amp.GradScaler("cuda",enabled=amp_enabled)


Final refit target selected from dev: epoch_frac=3.000 | selected_batch=2106 | selected_update=3157
Prefix K/V initialized from frozen TRAIN encoder activations | {'records': 256, 'key_abs_mean': 4.065642833709717, 'value_abs_mean': 1.6514452695846558, 'key_std': 5.262839317321777, 'value_std': 2.3491320610046387}
tokenized 18987 records in 0.3s
AdamW fused: True
Refit target: 7122 micro-batches total | loader batches/epoch: 2374 | full epochs: 3 | remainder batches: 0


In [19]:
# ============================================================
# CELL 19 — Final Prefix refit with exact fractional replay + ETA
# ============================================================
final_optimizer.zero_grad(set_to_none=True)
refit_start=time.time()
refit_updates=0
seen_batches=0
done=False

max_refit_epochs=int(math.ceil(REFIT_TOTAL_BATCHES/len(all_loader)))+1

for epoch in range(1,max_refit_epochs+1):
    if hasattr(all_loader.batch_sampler,"set_epoch"):
        all_loader.batch_sampler.set_epoch(epoch)

    final_model.train()
    running=torch.zeros((),device=DEVICE,dtype=torch.float32)
    n_batches=0
    ep0=time.time()

    for step,batch in enumerate(all_loader,1):
        if seen_batches>=REFIT_TOTAL_BATCHES:
            done=True; break

        labels=batch["labels"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast(
            device_type="cuda",dtype=torch.float16,enabled=amp_enabled
        ):
            logits=_forward_batch(final_model,batch)
            loss=final_criterion(logits,labels)
            scaled=loss/GRAD_ACCUM_STEPS

        if step%NAN_CHECK_EVERY==0:
            if not bool(torch.isfinite(loss.detach()).item()):
                raise FloatingPointError(
                    f"Non-finite refit loss epoch={epoch} batch={step}"
                )

        final_scaler.scale(scaled).backward()
        running.add_(loss.detach().float())
        n_batches+=1
        seen_batches+=1

        do_update=(seen_batches%GRAD_ACCUM_STEPS==0) or (seen_batches>=REFIT_TOTAL_BATCHES)
        if do_update:
            final_scaler.unscale_(final_optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in final_model.parameters() if p.requires_grad],GRAD_CLIP
            )
            old_scale=final_scaler.get_scale()
            final_scaler.step(final_optimizer); final_scaler.update()
            new_scale=final_scaler.get_scale()
            final_optimizer.zero_grad(set_to_none=True)
            if new_scale>=old_scale:
                final_scheduler.step(); refit_updates+=1

        if seen_batches%PRINT_EVERY==0 or seen_batches>=REFIT_TOTAL_BATCHES:
            elapsed=time.time()-refit_start
            rate=elapsed/max(seen_batches,1)
            eta=rate*(REFIT_TOTAL_BATCHES-seen_batches)
            print(
                f"refit batch={seen_batches}/{REFIT_TOTAL_BATCHES} "
                f"epoch≈{seen_batches/len(all_loader):.3f} "
                f"loss={float((running/max(n_batches,1)).item()):.4f} "
                f"elapsed={fmt_sec(elapsed)} ETA_refit≈{fmt_sec(eta)}",
                flush=True
            )

        if seen_batches>=REFIT_TOTAL_BATCHES:
            done=True; break

    if done: break

assert seen_batches==REFIT_TOTAL_BATCHES,(seen_batches,REFIT_TOTAL_BATCHES)

final_task_state={
    k:v.detach().cpu()
    for k,v in final_model.state_dict().items()
    if k.startswith("prefix_module.") or k.startswith(HEAD_PREFIXES)
}
torch.save({
    "task_state":final_task_state,
    "source_best_epoch_frac":BEST_FRAC,
    "source_best_batch_in_epoch":BEST_BATCH,
    "source_best_global_update":BEST_UPDATE,
    "refit_total_batches":REFIT_TOTAL_BATCHES,
    "refit_optimizer_updates":refit_updates,
    "source_best_dev_f1":best_f1,
    "labels":LABELS,
    "label_to_id":LABEL_TO_ID,
    "config":{
        "experiment":"Phase3 Exp4 Model3 Prefix final refit",
        "prefix_type":"encoder_self_attention_KV",
        "prefix_len":PREFIX_LEN,
        "prefix_dropout":PREFIX_DROPOUT,
    },
},FINAL_CKPT)

print(
    "✅ Final refit saved:",FINAL_CKPT,
    "| total refit:",fmt_sec(time.time()-refit_start)
)


refit batch=250/7122 epoch≈0.105 loss=2.3389 elapsed=00:00:06 ETA_refit≈00:03:00
refit batch=500/7122 epoch≈0.211 loss=2.2246 elapsed=00:00:13 ETA_refit≈00:02:56
refit batch=750/7122 epoch≈0.316 loss=2.1775 elapsed=00:00:20 ETA_refit≈00:02:51
refit batch=1000/7122 epoch≈0.421 loss=2.1305 elapsed=00:00:26 ETA_refit≈00:02:44
refit batch=1250/7122 epoch≈0.527 loss=2.0962 elapsed=00:00:33 ETA_refit≈00:02:37
refit batch=1500/7122 epoch≈0.632 loss=2.0742 elapsed=00:00:40 ETA_refit≈00:02:30
refit batch=1750/7122 epoch≈0.737 loss=2.0466 elapsed=00:00:46 ETA_refit≈00:02:22
refit batch=2000/7122 epoch≈0.842 loss=2.0278 elapsed=00:00:53 ETA_refit≈00:02:15
refit batch=2250/7122 epoch≈0.948 loss=2.0092 elapsed=00:00:59 ETA_refit≈00:02:08
refit batch=2500/7122 epoch≈1.053 loss=1.9001 elapsed=00:01:05 ETA_refit≈00:02:01
refit batch=2750/7122 epoch≈1.158 loss=1.8872 elapsed=00:01:12 ETA_refit≈00:01:55
refit batch=3000/7122 epoch≈1.264 loss=1.8621 elapsed=00:01:18 ETA_refit≈00:01:48
refit batch=3250/71

In [21]:
# ============================================================
# CELL 20 — Build test sentence records
# ============================================================

def make_test_records(test_dict, domain):
    records = []
    for raw_id, sents in test_dict.items():
        did = normalize_id(raw_id)
        assert isinstance(sents, list)
        for i, sent in enumerate(sents):
            records.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
            })
    return records

test_records = (
    make_test_records(cl_test_raw, "CL")
    + make_test_records(it_test_raw, "IT")
)

test_loader = make_loader(test_records, False, EVAL_BATCH_SIZE, False)

print("Total test sentence records:", len(test_records))
assert len(test_records) == (
    sum(len(v) for v in cl_test_raw.values())
    + sum(len(v) for v in it_test_raw.values())
)


tokenized 2197 records in 0.0s
Total test sentence records: 2197


In [22]:
# ============================================================
# CELL 21 — Test inference + selected transition decoding
# Variants: heldout (best dev ckpt), refit (train+dev), ensemble (mean log-prob)
# ============================================================
domain_stats_all=build_domain_stats(cl_train_raw+cl_dev_raw,it_train_raw+it_dev_raw)
def _logp(lg): return log_softmax_np(lg.astype(np.float64))
def decode_to_prediction(lg,meta,name):
    ids=decode_docs(lg,meta,domain_stats_all,DECODE_ALPHA,DECODE_BETA); pred=records_to_prediction(meta,ids); cnt=Counter(ids.tolist())
    print(f"[{name}] predicted label counts:",{LABELS[k]:cnt[k] for k in sorted(cnt)}); return pred

lg_held,meta_held,_=predict_logits(model,test_loader)
lg_refit,meta_refit,_=predict_logits(final_model,test_loader)
assert meta_held==meta_refit
np.savez_compressed(
    ROOT/'test_logits.npz',
    heldout=lg_held,
    refit=lg_refit,
    doc_id=np.asarray([x[0] for x in meta_held]),
    sent_idx=np.asarray([x[1] for x in meta_held],dtype=np.int64),
    domain=np.asarray([x[2] for x in meta_held]),
)

predictions={'heldout':decode_to_prediction(lg_held,meta_held,'heldout'),
             'refit':decode_to_prediction(lg_refit,meta_refit,'refit')}
if ENSEMBLE_HELDOUT_AND_REFIT:
    predictions['ensemble']=decode_to_prediction(0.5*(_logp(lg_held)+_logp(lg_refit)),meta_held,'ensemble')
prediction=predictions['refit']
with open(OFFICIAL_PRED,'w',encoding='utf-8') as f: json.dump(prediction,f,ensure_ascii=False,indent=2)
print('Saved:',OFFICIAL_PRED,'Prediction documents:',len(prediction))


[heldout] predicted label counts: {'Fact': 524, 'Issue': 5, 'ArgumentPetitioner': 6, 'ArgumentRespondent': 56, 'Statute': 278, 'Dissent': 138, 'PrecedentReliedUpon': 71, 'PrecedentNotReliedUpon': 48, 'PrecedentOverruled': 667, 'RulingByLowerCourt': 5, 'RatioOfTheDecision': 298, 'RulingByPresentCourt': 79, 'None': 22}
[refit] predicted label counts: {'Fact': 503, 'Issue': 8, 'ArgumentPetitioner': 104, 'ArgumentRespondent': 34, 'Statute': 253, 'Dissent': 126, 'PrecedentReliedUpon': 192, 'PrecedentNotReliedUpon': 14, 'PrecedentOverruled': 419, 'RulingByLowerCourt': 2, 'RatioOfTheDecision': 440, 'RulingByPresentCourt': 92, 'None': 10}
[ensemble] predicted label counts: {'Fact': 517, 'Issue': 4, 'ArgumentPetitioner': 43, 'ArgumentRespondent': 43, 'Statute': 264, 'Dissent': 139, 'PrecedentReliedUpon': 129, 'PrecedentNotReliedUpon': 15, 'PrecedentOverruled': 542, 'RulingByLowerCourt': 5, 'RatioOfTheDecision': 392, 'RulingByPresentCourt': 88, 'None': 16}
Saved: /kaggle/working/phase3_exp4_mode

In [23]:
# ============================================================
# CELL 22 — Strict submission validation + ZIP
# Mirrors evaluator structural rules for private test data.
# ============================================================

def normalized_test_map(raw):
    out = {}
    for raw_id, sents in raw.items():
        did = normalize_id(raw_id)
        assert did not in out, f"Duplicate normalized ID: {did}"
        out[did] = sents
    return out

cl_test = normalized_test_map(cl_test_raw)
it_test = normalized_test_map(it_test_raw)

assert not (set(cl_test) & set(it_test)), "CL/IT test IDs overlap after normalization."

expected = {}
expected.update(cl_test)
expected.update(it_test)

assert set(prediction) == set(expected), (
    f"ID mismatch: missing={set(expected)-set(prediction)}, "
    f"extra={set(prediction)-set(expected)}"
)

allowed = set(LABELS)
for doc_id, sents in expected.items():
    pred = prediction[doc_id]
    assert isinstance(pred, list), doc_id
    assert len(pred) == len(sents), (
        doc_id, len(pred), len(sents)
    )
    assert all(x in allowed for x in pred), doc_id

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    z.write(OFFICIAL_PRED, arcname="prediction.json")

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as z:
    names = z.namelist()
    assert names == ["prediction.json"], names
    zipped = json.loads(z.read("prediction.json").decode("utf-8"))

assert zipped == prediction

print("✅ FINAL SUBMISSION PASSED STRUCTURAL CHECKS")
print("Documents:", len(prediction))
print("Sentences:", sum(len(v) for v in prediction.values()))
print("ZIP contents:", names)
print("prediction.json:", OFFICIAL_PRED)
print("submission.zip :", SUBMISSION_ZIP)

# ---- also write the two variants separately (upload both; keep the better)
def write_variant(name, pred):
    assert set(pred) == set(expected)
    for doc_id, sents in expected.items():
        assert len(pred[doc_id]) == len(sents), (name, doc_id)
        assert all(x in allowed for x in pred[doc_id]), (name, doc_id)
    jp = ROOT / f"prediction_{name}.json"
    zp = ROOT / f"submission_{name}.zip"
    with open(jp, "w", encoding="utf-8") as f:
        json.dump(pred, f, ensure_ascii=False)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    print("✅", zp.name)

for name, pred in predictions.items():
    write_variant(name, pred)
print("Variants written:", ", ".join(f"submission_{k}.zip" for k in predictions), "| submission.zip == refit")

# Sanity: how much do the variants disagree at sentence level?
def _flat(p): return [x for d in sorted(expected) for x in p[d]]
_names = list(predictions)
for i in range(len(_names)):
    for j in range(i + 1, len(_names)):
        a, b = _flat(predictions[_names[i]]), _flat(predictions[_names[j]])
        print(f"agreement {_names[i]} vs {_names[j]}: {np.mean([x == y for x, y in zip(a, b)]):.4f}")


# Evaluator-style structural contract check on TEST.
# Test gold labels are private, so only IDs, lengths and allowed labels can be verified.
def test_submission_contract(prediction_obj):
    if not isinstance(prediction_obj, dict):
        raise ValueError(
            "prediction.json must be a JSON object mapping document IDs to label lists."
        )

    normalized = {}

    for raw_id, labels in prediction_obj.items():
        did = normalize_id(raw_id)

        if did in normalized:
            raise ValueError(
                f"Duplicate document ID after .txt normalization: {did}"
            )

        normalized[did] = labels

    if set(normalized) != set(expected):
        missing = sorted(set(expected) - set(normalized))
        extra = sorted(set(normalized) - set(expected))
        raise ValueError(
            f"ID mismatch: missing={len(missing)}, extra={len(extra)}"
        )

    for did, source_sentences in expected.items():
        pred = normalized[did]

        if not isinstance(pred, list):
            raise ValueError(f"{did}: labels must be a JSON list.")

        if len(pred) != len(source_sentences):
            raise ValueError(
                f"{did}: sentence-count mismatch "
                f"({len(pred)} vs {len(source_sentences)})."
            )

        if any(x not in allowed for x in pred):
            raise ValueError(f"{did}: unknown label encountered.")

    return True

assert test_submission_contract(prediction)
print(
    "✅ TEST prediction satisfies the same ID / sentence-count / "
    "allowed-label contract enforced by the Phase-3 evaluator."
)



# Compact reproducibility manifest for future experiments/reporting.
manifest={
    "experiment":"Phase3_Exp4_Model3_PrefixTuning",
    "phase1_model":"Model3_EncoderDecoder",
    "technique":"Prefix Tuning",
    "prefix_type":"encoder_self_attention_KV",
    "prefix_len":PREFIX_LEN,
    "prefix_dropout":PREFIX_DROPOUT,
    "prefix_init_from_tokens":PREFIX_INIT_FROM_TOKENS,
"prefix_init_mode":PREFIX_INIT_MODE,
"prefix_init_records":PREFIX_INIT_RECORDS if PREFIX_INIT_FROM_TOKENS else 0,
"prefix_init_stats":PREFIX_INIT_STATS,
    "prefix_trainable_params":2*N_ENCODER_LAYERS*PREFIX_LEN*D_MODEL,
    "seed":SEED,
    "max_len":MAX_LEN,
    "context_sents_each_side":CONTEXT_SENTS,
    "micro_batch_size":MICRO_BATCH_SIZE,
    "grad_accum_steps":GRAD_ACCUM_STEPS,
    "effective_batch_size":MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS,
    "prefix_lr":PREFIX_LR,
    "head_lr":HEAD_LR,
    "best_epoch":int(best_obj["best_epoch"]),
    "best_epoch_frac":float(best_obj.get("best_frac",best_obj["best_epoch"])),
    "best_batch_in_epoch":int(best_obj.get("best_batch_in_epoch",0)),
    "best_global_update":int(best_obj.get("best_global_update",0)),
    "best_dev_macro_f1":float(best_obj["best_macro_f1"]),
    "decode_alpha":float(DECODE_ALPHA),
    "decode_beta":float(DECODE_BETA),
    "primary_submission":"submission.zip == refit",
    "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
    "phase1_checkpoint":str(PHASE1_CKPT),
    "artifacts":sorted([p.name for p in ROOT.iterdir() if p.is_file()]),
}
with open(ROOT/"experiment_manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2)
print("✅ Saved experiment manifest:",ROOT/"experiment_manifest.json")


✅ FINAL SUBMISSION PASSED STRUCTURAL CHECKS
Documents: 10
Sentences: 2197
ZIP contents: ['prediction.json']
prediction.json: /kaggle/working/phase3_exp4_model3_prefix/prediction.json
submission.zip : /kaggle/working/phase3_exp4_model3_prefix/submission.zip
✅ submission_heldout.zip
✅ submission_refit.zip
✅ submission_ensemble.zip
Variants written: submission_heldout.zip, submission_refit.zip, submission_ensemble.zip | submission.zip == refit
agreement heldout vs refit: 0.7360
agreement heldout vs ensemble: 0.8753
agreement refit vs ensemble: 0.8416
✅ TEST prediction satisfies the same ID / sentence-count / allowed-label contract enforced by the Phase-3 evaluator.
✅ Saved experiment manifest: /kaggle/working/phase3_exp4_model3_prefix/experiment_manifest.json


## What to record for Experiment 4

Download and keep:
- `best_prefix_rr.pt`
- `final_refit_prefix_rr.pt`
- `submission.zip`
- `submission_heldout.zip`
- `submission_refit.zip`
- `submission_ensemble.zip`
- `prediction_dev.json`
- `prediction_dev_decoded.json`
- `dev_logits.npz`, `test_logits.npz`
- `history.json`, `experiment_manifest.json`
- `cl_dev_truth.json`, `it_dev_truth.json`, `labels.json`, `evaluate_phase3.py`
- completed notebook

For the report record:
- Phase-1 architecture: **Model 3 encoder–decoder**
- Phase-3 technique: **true encoder self-attention K/V Prefix Tuning**
- prefix length = `20`
- all Phase-1 parameters frozen
- task classification uses the pretrained bidirectional encoder representations
- exact prefix/trainable parameter counts printed by Cell 10
- best heldout Macro-F1 and exact selected epoch fraction/batch/update
- selected dev transition-decoding setting
- Codabench heldout/refit/ensemble scores

### Run order
1. Attach `encdec_full_best*.pt` and exact `tokenizer.json`; enable T4.
2. Run top-to-bottom.
3. Confirm strict Phase-1 checkpoint loading succeeds.
4. Confirm original backbone trainable params = **0**.
5. Confirm Prefix params = **184,320** for prefix length 20.
6. Let heldout training and early stopping finish.
7. Confirm the **exact supplied Phase-3 evaluator passes before refit**.
8. Continue through decoding, fractional train+dev refit, test inference and structural checks.
9. If submission budget allows, test heldout/refit/ensemble separately; retain the highest Codabench Macro-F1 as Exp4's result.

### Optional future ablations (not part of this Exp4 run; change one thing at a time and compare DEV Macro-F1)
1. `PREFIX_INIT_FROM_TOKENS = False` to measure how much the real-activation init contributes.
2. `PREFIX_LR` 3e-4 → 1e-3 / 3e-3: direct (non-reparameterised) prefixes usually need a larger LR than adapters.
3. `PREFIX_LEN` 20 → 50 (the parameter-count assertions and the "184,320" note scale with it automatically).
4. `CONTEXT_SENTS` 1 → 2 with larger `PREV_BUDGET` / `NEXT_BUDGET`.
